# Árvore geradora mínima: Kruskal e Prim — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As verificações, os gráficos e alguns exercícios ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- calcular o peso de uma árvore geradora e encontrar a AGM por **força bruta** em grafos pequenos;
- encontrar a **aresta leve** de um corte e conferir experimentalmente o **teorema do corte**;
- implementar **union-find** com união por tamanho e compressão de caminho e usá-lo no algoritmo de **Kruskal**;
- implementar o algoritmo de **Prim** com heap binário e compará-lo com a versão $\Theta(n^2)$ com matriz;
- usar Kruskal interrompido para **agrupar** pontos com espaçamento máximo.

Os vértices do exemplo condutor (Cormen et al., Figura 23.1) são $a, b, \dots, i$; nos programas
em C eles viram os números $0, 1, \dots, 8$.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess, math, random, itertools

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True,
                       errors='replace')
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor de 'nome=valor' da saída (int se for inteiro, float se for número, senão str)."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            valor = tok.split('=', 1)[1]
            if valor.lstrip('-').isdigit():
                return int(valor)
            try:
                return float(valor)
            except ValueError:
                return valor
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

# Exemplo condutor: Cormen et al. (2009), §23.1, Figura 23.1.
VERT = 'abcdefghi'
ARESTAS = [('a', 'b', 4), ('a', 'h', 8), ('b', 'c', 8), ('b', 'h', 11), ('c', 'd', 7),
           ('c', 'f', 4), ('c', 'i', 2), ('d', 'e', 9), ('d', 'f', 14), ('e', 'f', 10),
           ('f', 'g', 2), ('g', 'h', 1), ('g', 'i', 6), ('h', 'i', 7)]
NUM = {c: k for k, c in enumerate(VERT)}            # 'a' -> 0, ..., 'i' -> 8

def entrada_grafo(n, arestas):
    """Formato lido pelos programas em C: 'n m' e depois m linhas 'u v w' (vértices 0..n-1)."""
    linhas = [f'{n} {len(arestas)}'] + [f'{NUM.get(u, u)} {NUM.get(v, v)} {w}' for u, v, w in arestas]
    return '\n'.join(linhas) + '\n'

# Posições dos vértices no desenho (as mesmas dos slides).
POS = {'a': (0, 0), 'b': (1, 1), 'c': (2.3, 1), 'd': (3.6, 1), 'e': (4.6, 0),
       'f': (3.6, -1), 'g': (2.3, -1), 'h': (1, -1), 'i': (1.75, 0)}

In [ ]:
# Desenho do grafo condutor com um conjunto de arestas destacado.
import matplotlib.pyplot as plt

def desenha(destaque=(), titulo='', ax=None, lado=()):
    """destaque: arestas (u, v) em laranja; lado: vértices pintados (por exemplo, um lado S de um corte)."""
    destaque = {frozenset(e[:2]) for e in destaque}
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 2.8))
    for u, v, w in ARESTAS:
        (x1, y1), (x2, y2) = POS[u], POS[v]
        dentro = frozenset((u, v)) in destaque
        ax.plot([x1, x2], [y1, y2], color='darkorange' if dentro else 'lightgray',
                linewidth=4 if dentro else 1, zorder=1)
        ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(w), fontsize=8, ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none'), zorder=2)
    for c, (x, y) in POS.items():
        ax.scatter(x, y, s=330, zorder=3, color='navajowhite' if c in lado else 'lightsteelblue',
                   edgecolor='darkorange' if c in lado else 'steelblue')
        ax.text(x, y, c, ha='center', va='center', fontsize=9, zorder=4)
    ax.set_title(titulo); ax.axis('off'); ax.set_aspect('equal'); ax.margins(0.08)
    return ax

desenha(titulo='O grafo condutor (Cormen et al., Figura 23.1)')
plt.show()

## 1. Árvores geradoras e o teorema do corte

Uma **árvore geradora** de um grafo conexo $G = (V, E)$ é um conjunto de $n - 1$ arestas sem
ciclos que liga todos os $n$ vértices; a **AGM** é a de menor peso total. Em grafos pequenos dá
para testar todos os subconjuntos de $n - 1$ arestas: no exemplo condutor são
$\binom{14}{8} = 3003$. Um subconjunto com $n - 1$ arestas é árvore geradora exatamente quando
não fecha ciclo, o que conferimos com uma estrutura simples de componentes.

In [ ]:
# Exemplo: AGM por força bruta no grafo condutor.
def sem_ciclo(n, arestas):
    """True se as arestas (u, v, w) não fecham ciclo; vértices são índices 0..n-1."""
    comp = list(range(n))
    for u, v, _ in arestas:
        if comp[u] == comp[v]:
            return False
        velho = comp[v]
        comp = [comp[u] if c == velho else c for c in comp]
    return True

def agm_forca_bruta(n, arestas):
    """Devolve (peso mínimo, lista de todas as árvores geradoras de peso mínimo)."""
    melhor, otimas, total = math.inf, [], 0
    for sub in itertools.combinations(arestas, n - 1):
        if sem_ciclo(n, sub):                 # n-1 arestas sem ciclo = árvore geradora
            total += 1
            p = sum(w for *_, w in sub)
            if p < melhor:
                melhor, otimas = p, [sub]
            elif p == melhor:
                otimas.append(sub)
    agm_forca_bruta.arvores = total
    return melhor, otimas

ARESTAS_NUM = [(NUM[u], NUM[v], w) for u, v, w in ARESTAS]
peso, otimas = agm_forca_bruta(9, ARESTAS_NUM)
print(f'{agm_forca_bruta.arvores} árvores geradoras; peso mínimo {peso}; {len(otimas)} AGMs:')
for t in otimas:
    print('  ', ' '.join(f'{VERT[u]}{VERT[v]}' for u, v, _ in t))
assert peso == 37

In [ ]:
# Visualização: todas as AGMs do grafo condutor.
fig, axs = plt.subplots(1, len(otimas), figsize=(6 * len(otimas), 2.8))
for ax, t in zip(axs if len(otimas) > 1 else [axs], otimas):
    desenha([(VERT[u], VERT[v]) for u, v, _ in t], f'peso {sum(w for *_, w in t)}', ax=ax)
plt.show()

Um **corte** $(S, V - S)$ divide os vértices em dois lados; uma aresta o **cruza** se tem uma
ponta em cada lado, e é **leve** se tem o menor peso entre as que cruzam. O **teorema do corte**
garante que uma aresta leve é segura. Quando os pesos são todos distintos, a AGM é única e
**toda** aresta leve de **qualquer** corte pertence a ela.

### Exercício 1: a aresta leve de um corte

Complete `aresta_leve(arestas, S)`, que recebe a lista de arestas `(u, v, w)` e um conjunto de
vértices `S` e devolve a aresta `(u, v, w)` de menor peso que cruza o corte $(S, V - S)$, ou
`None` se nenhuma aresta cruza. O teste confere o corte dos slides e verifica o teorema do corte
em 200 grafos aleatórios com pesos distintos, comparando com a AGM da força bruta.

In [ ]:
# Exercício 1: aresta leve de um corte
# TODO: implemente aresta_leve(arestas, S).

def aresta_leve(arestas, S):
    # TODO: implemente aqui
    pass


# Teste automático
S = {'a', 'b', 'h', 'i', 'g'}
assert aresta_leve(ARESTAS, S)[2] == 2, 'Slides: as arestas leves de S = {a,b,h,i,g} pesam 2'
assert aresta_leve(ARESTAS, set(VERT)) is None, 'Se S = V, nenhuma aresta cruza o corte'
assert aresta_leve(ARESTAS, {'e'})[:2] in {('d', 'e'), ('e', 'd')}, 'O corte {e} tem leve (d,e)'

def grafo_aleatorio(n, extra, rng):
    """Grafo conexo: uma árvore aleatória mais 'extra' arestas, com pesos distintos."""
    pares = {(rng.randrange(v), v) for v in range(1, n)}
    extra = min(extra, n * (n - 1) // 2 - (n - 1))   # não passar do grafo completo
    while len(pares) < n - 1 + extra:
        u, v = sorted(rng.sample(range(n), 2))
        pares.add((u, v))
    pesos = rng.sample(range(1, 100), len(pares))
    return [(u, v, w) for (u, v), w in zip(sorted(pares), pesos)]

rng = random.Random(2026)
for _ in range(200):
    n = rng.randint(3, 6)
    arestas = grafo_aleatorio(n, rng.randint(0, n), rng)
    _, (agm,) = agm_forca_bruta(n, arestas)   # pesos distintos: AGM única
    S = set(rng.sample(range(n), rng.randint(1, n - 1)))
    e = aresta_leve(arestas, S)
    assert e in agm, f'Aresta leve {e} do corte {S} fora da AGM {agm}'
print('Exercício 1 ok! A aresta leve estava na AGM em 200 cortes aleatórios.')

## 2. O algoritmo de Kruskal

Kruskal ordena as arestas por peso e as percorre uma vez: aceita a aresta se ela liga dois
**componentes diferentes** da floresta já escolhida e rejeita se fecha ciclo. A aresta aceita é
leve no corte "componente de $u$ contra o resto", logo é segura. A versão abaixo guarda o
**rótulo** do componente de cada vértice: comparar rótulos é $O(1)$, mas cada fusão percorre o
vetor inteiro, $\Theta(n)$.

In [ ]:
%%writefile src/kruskal.c
/* Algoritmo de KRUSKAL com componentes rotulados.
   Uso: ./kruskal < entrada          (imprime cada decisao e o peso da AGM)
        ./kruskal tempo n m          (grafo aleatorio; imprime o peso e o tempo)
   Kruskal (1956); Cormen et al. (2009), sec. 23.2; Levitin (2012), sec. 9.2. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

typedef struct { int u, v, w; } Aresta;

/* le "n m" e depois m linhas "u v w" (vertices 0..n-1) */
Aresta *le_grafo( int *n, int *m) {
   if (scanf( "%d %d", n, m) != 2) return NULL;
   Aresta *e = malloc( *m * sizeof (Aresta));
   for (int i = 0; i < *m; ++i)
      if (scanf( "%d %d %d", &e[i].u, &e[i].v, &e[i].w) != 3) return NULL;
   return e;
}

/* gerador pseudoaleatorio xorshift (o mesmo em todos os programas do tutorial) */
unsigned long long semente = 88172645463325252ULL;
unsigned aleat( void) {
   semente ^= semente << 13;  semente ^= semente >> 7;  semente ^= semente << 17;
   return (unsigned) (semente >> 11);
}

/* grafo conexo aleatorio: arvore aleatoria + m-(n-1) arestas quaisquer, pesos em 1..10^6 */
Aresta *grafo_aleatorio( int n, int m) {
   Aresta *e = malloc( m * sizeof (Aresta));
   for (int v = 1; v < n; ++v) {
      e[v-1].u = (int) (aleat() % v);  e[v-1].v = v;  e[v-1].w = 1 + aleat() % 1000000;
   }
   for (int i = n - 1; i < m; ++i) {
      int u = aleat() % n, v = aleat() % n;
      if (u == v) v = (v + 1) % n;
      e[i].u = u;  e[i].v = v;  e[i].w = 1 + aleat() % 1000000;
   }
   return e;
}

/* ordem crescente de peso; empate pelas pontas (reproduz a ordem dos slides) */
int cmp_peso( const void *a, const void *b) {
   const Aresta *x = a, *y = b;
   if (x->w != y->w) return x->w < y->w ? -1 : 1;
   if (x->u != y->u) return x->u - y->u;
   return x->v - y->v;
}

double agora( void) { return (double) clock() / CLOCKS_PER_SEC; }

/* KRUSKAL com componentes rotulados ("quick-find"): comp[x] e o rotulo do componente de x.
   Testar se u e v estao juntos custa O(1); unir custa Theta(n), pois reetiqueta um componente.
   Grava em T[] as arestas da AGM e devolve quantas sao. */
int kruskal_qf( int n, int m, Aresta e[], Aresta T[], int mostra) {
   qsort( e, m, sizeof (Aresta), cmp_peso);
   int *comp = malloc( n * sizeof (int));
   for (int v = 0; v < n; ++v) comp[v] = v;
   int k = 0;
   for (int i = 0; i < m && k < n - 1; ++i) {
      int a = comp[e[i].u], b = comp[e[i].v];
      if (a != b) {                          /* componentes distintos: aceita */
         T[k++] = e[i];
         for (int x = 0; x < n; ++x)         /* funde: rotulo b vira a */
            if (comp[x] == b) comp[x] = a;
         if (mostra) printf( "aceita %d %d %d\n", e[i].u, e[i].v, e[i].w);
      } else if (mostra)
         printf( "rejeita %d %d %d\n", e[i].u, e[i].v, e[i].w);
   }
   free( comp);
   return k;
}

int main( int argc, char *argv[]) {
   int n, m, tempo = argc == 4 && strcmp( argv[1], "tempo") == 0;
   Aresta *e;
   if (tempo) { n = atoi( argv[2]);  m = atoi( argv[3]);  e = grafo_aleatorio( n, m); }
   else if ((e = le_grafo( &n, &m)) == NULL) return 1;
   Aresta *T = malloc( n * sizeof (Aresta));
   double t0 = agora();
   int k = kruskal_qf( n, m, e, T, !tempo);
   double t = agora() - t0;
   long peso = 0;
   for (int i = 0; i < k; ++i) peso += T[i].w;
   printf( "arestas=%d peso=%ld", k, peso);
   if (tempo) printf( " t=%.6f", t);
   printf( "\n");
   return 0;
}

In [ ]:
# Exemplo: Kruskal no grafo condutor (compare com a tabela dos slides).
saida = rodar('kruskal.c', entrada=entrada_grafo(9, ARESTAS), mostrar=False)
aceitas = []
for lin in saida.splitlines():
    if lin.startswith(('aceita', 'rejeita')):
        dec, u, v, w = lin.split()
        print(f'{dec:8s} ({VERT[int(u)]},{VERT[int(v)]}) {w}')
        if dec == 'aceita':
            aceitas.append((VERT[int(u)], VERT[int(v)]))
print(saida.splitlines()[-1])
assert campo(saida, 'peso') == 37 and campo(saida, 'arestas') == 8
desenha(aceitas, 'Kruskal: a AGM com (a,h)')
plt.show()

### Exercício 2: union-find no Kruskal

Com rótulos, as $n - 1$ fusões custam $\Theta(n^2)$ no total. Troque-os por uma **floresta de
conjuntos disjuntos**: `pai[x]` aponta para o pai de `x`, e a raiz (com `pai[r] == r`) é o
representante. Complete:

- `find(x)`: devolve a raiz da árvore de `x` e faz **compressão de caminho** (todos os vértices
  do caminho passam a apontar direto para a raiz);
- `une(a, b)`: recebe duas **raízes** distintas e pendura a de menor `tam[]` na outra
  (**união por tamanho**), atualizando `tam[]`.

O `kruskal` já está escrito e usa as suas funções. O `main` confere a compressão de caminho, a
altura das árvores com a união por tamanho, o grafo condutor e 300 grafos aleatórios contra a
versão com rótulos.

In [ ]:
%%writefile src/kruskal_uf.c
/* Exercicio 2: UNION-FIND (uniao por tamanho + compressao de caminho) no Kruskal.
   Uso: ./kruskal_uf               (verificacao)
        ./kruskal_uf tempo n m     (grafo aleatorio; imprime o peso e o tempo)
   TODO: complete find() e une(). */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

typedef struct { int u, v, w; } Aresta;

/* le "n m" e depois m linhas "u v w" (vertices 0..n-1) */
Aresta *le_grafo( int *n, int *m) {
   if (scanf( "%d %d", n, m) != 2) return NULL;
   Aresta *e = malloc( *m * sizeof (Aresta));
   for (int i = 0; i < *m; ++i)
      if (scanf( "%d %d %d", &e[i].u, &e[i].v, &e[i].w) != 3) return NULL;
   return e;
}

/* gerador pseudoaleatorio xorshift (o mesmo em todos os programas do tutorial) */
unsigned long long semente = 88172645463325252ULL;
unsigned aleat( void) {
   semente ^= semente << 13;  semente ^= semente >> 7;  semente ^= semente << 17;
   return (unsigned) (semente >> 11);
}

/* grafo conexo aleatorio: arvore aleatoria + m-(n-1) arestas quaisquer, pesos em 1..10^6 */
Aresta *grafo_aleatorio( int n, int m) {
   Aresta *e = malloc( m * sizeof (Aresta));
   for (int v = 1; v < n; ++v) {
      e[v-1].u = (int) (aleat() % v);  e[v-1].v = v;  e[v-1].w = 1 + aleat() % 1000000;
   }
   for (int i = n - 1; i < m; ++i) {
      int u = aleat() % n, v = aleat() % n;
      if (u == v) v = (v + 1) % n;
      e[i].u = u;  e[i].v = v;  e[i].w = 1 + aleat() % 1000000;
   }
   return e;
}

/* ordem crescente de peso; empate pelas pontas (reproduz a ordem dos slides) */
int cmp_peso( const void *a, const void *b) {
   const Aresta *x = a, *y = b;
   if (x->w != y->w) return x->w < y->w ? -1 : 1;
   if (x->u != y->u) return x->u - y->u;
   return x->v - y->v;
}

double agora( void) { return (double) clock() / CLOCKS_PER_SEC; }

#define MAXN 200000
int pai[MAXN], tam[MAXN];

/* devolve a raiz da arvore de x, com compressao de caminho */
int find( int x) {
   /* TODO */
   return x;
}

/* a e b sao raizes distintas: pendura a de menor tam[] na outra */
void une( int a, int b) {
   /* TODO */
}

/* ---------- Kruskal com union-find (nao altere) ---------- */
int kruskal_uf( int n, int m, Aresta e[], Aresta T[]) {
   qsort( e, m, sizeof (Aresta), cmp_peso);
   for (int v = 0; v < n; ++v) { pai[v] = v;  tam[v] = 1; }
   int k = 0;
   for (int i = 0; i < m && k < n - 1; ++i) {
      int a = find( e[i].u), b = find( e[i].v);
      if (a != b) {
         T[k++] = e[i];
         une( a, b);
      }
   }
   return k;
}

/* KRUSKAL com componentes rotulados ("quick-find"): comp[x] e o rotulo do componente de x.
   Testar se u e v estao juntos custa O(1); unir custa Theta(n), pois reetiqueta um componente.
   Grava em T[] as arestas da AGM e devolve quantas sao. */
int kruskal_qf( int n, int m, Aresta e[], Aresta T[], int mostra) {
   qsort( e, m, sizeof (Aresta), cmp_peso);
   int *comp = malloc( n * sizeof (int));
   for (int v = 0; v < n; ++v) comp[v] = v;
   int k = 0;
   for (int i = 0; i < m && k < n - 1; ++i) {
      int a = comp[e[i].u], b = comp[e[i].v];
      if (a != b) {                          /* componentes distintos: aceita */
         T[k++] = e[i];
         for (int x = 0; x < n; ++x)         /* funde: rotulo b vira a */
            if (comp[x] == b) comp[x] = a;
         if (mostra) printf( "aceita %d %d %d\n", e[i].u, e[i].v, e[i].w);
      } else if (mostra)
         printf( "rejeita %d %d %d\n", e[i].u, e[i].v, e[i].w);
   }
   free( comp);
   return k;
}

/* ---------- verificacao (nao altere) ---------- */
static long peso_total( int k, Aresta T[]) {
   long p = 0;
   for (int i = 0; i < k; ++i) p += T[i].w;
   return p;
}

static int altura( int x) {            /* sobe sem comprimir */
   int h = 0;
   while (pai[x] != x) { x = pai[x];  ++h; }
   return h;
}

static int erro( const char *msg) { printf( "ERRO: %s\nok=0\n", msg);  return 0; }

int main( int argc, char *argv[]) {
   if (argc == 4 && strcmp( argv[1], "tempo") == 0) {
      int n = atoi( argv[2]), m = atoi( argv[3]);
      Aresta *e = grafo_aleatorio( n, m), *T = malloc( n * sizeof (Aresta));
      double t0 = agora();
      long p = peso_total( kruskal_uf( n, m, e, T), T);
      printf( "peso=%ld t=%.6f\n", p, agora() - t0);
      return 0;
   }
   /* 1. compressao de caminho: cadeia 0 -> 1 -> ... -> 9 */
   for (int x = 0; x < 10; ++x) { pai[x] = x + 1;  tam[x] = 1; }
   pai[9] = 9;  tam[9] = 10;
   if (find( 0) != 9) return erro( "find(0) deveria devolver a raiz 9");
   for (int x = 0; x < 9; ++x)
      if (pai[x] != 9) return erro( "depois de find(0), todo o caminho deve apontar para 9");
   /* 2. uniao por tamanho: juntar sempre um vertice isolado ao conjunto grande */
   int n = 1 << 12;
   for (int x = 0; x < n; ++x) { pai[x] = x;  tam[x] = 1; }
   for (int x = 1; x < n; ++x) une( find( x), find( 0));
   int r = find( 0), hmax = 0;
   if (tam[r] != n) return erro( "tam[] da raiz deveria ser n depois de unir tudo");
   for (int x = 0; x < n; ++x) if (altura( x) > hmax) hmax = altura( x);
   if (hmax > 12) return erro( "altura maior que log2(n): a menor arvore deve ir para baixo");
   /* 3. grafo condutor */
   int ex[14][3] = {{0,1,4},{0,7,8},{1,2,8},{1,7,11},{2,3,7},{2,5,4},{2,8,2},
                    {3,4,9},{3,5,14},{4,5,10},{5,6,2},{6,7,1},{6,8,6},{7,8,7}};
   Aresta e[14], T[9];
   for (int i = 0; i < 14; ++i) { e[i].u = ex[i][0];  e[i].v = ex[i][1];  e[i].w = ex[i][2]; }
   long p = peso_total( kruskal_uf( 9, 14, e, T), T);
   printf( "peso=%ld\n", p);
   if (p != 37) return erro( "o grafo condutor tem AGM de peso 37");
   /* 4. grafos aleatorios contra a versao com rotulos */
   for (int t = 0; t < 300; ++t) {
      int nn = 2 + aleat() % 200, mm = nn - 1 + aleat() % (3 * nn);
      Aresta *a = grafo_aleatorio( nn, mm), *b = malloc( mm * sizeof (Aresta));
      Aresta *T1 = malloc( nn * sizeof (Aresta)), *T2 = malloc( nn * sizeof (Aresta));
      memcpy( b, a, mm * sizeof (Aresta));
      int k1 = kruskal_uf( nn, mm, a, T1), k2 = kruskal_qf( nn, mm, b, T2, 0);
      if (k1 != nn - 1 || peso_total( k1, T1) != peso_total( k2, T2))
         return erro( "peso diferente da versao com rotulos num grafo aleatorio");
      free( a);  free( b);  free( T1);  free( T2);
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 2
saida = rodar('kruskal_uf.c')
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 2 ok!')

In [ ]:
# Visualização: rótulos (Theta(n) por fusão) contra union-find, em grafos esparsos com m = 4n.
ns = [2**k for k in range(10, 16)]
t_qf = [campo(rodar('kruskal.c', args=('tempo', n, 4 * n), mostrar=False), 't') for n in ns]
t_uf = [campo(rodar('kruskal_uf.c', args=('tempo', n, 4 * n), mostrar=False), 't') for n in ns]

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(ns, t_qf, 'o-', label='rótulos: O(m log m + n²)')
ax.plot(ns, t_uf, 's-', label='union-find: O(m log m)')
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('n (m = 4n)'); ax.set_ylabel('tempo (s)'); ax.legend()
ax.set_title('Kruskal: o custo das fusões')
plt.show()

## 3. O algoritmo de Prim

Prim cresce **uma** árvore a partir do vértice $0$. Para cada vértice $v$ de fora, $\mathit{chave}[v]$
é o peso da aresta mais leve que liga $v$ à árvore, e $\mathit{pai}[v]$ é a outra ponta. Em cada
passo entra o vértice de menor chave, e as chaves dos seus vizinhos podem baixar. Com uma
matriz de adjacência e uma varredura para achar o mínimo, o custo é $\Theta(n^2)$ — ótimo para
grafos densos.

In [ ]:
%%writefile src/prim_matriz.c
/* Algoritmo de PRIM com matriz de adjacencia e varredura: Theta(n^2).
   Uso: ./prim_matriz < entrada         (imprime a ordem de entrada dos vertices e o peso)
        ./prim_matriz tempo n m         (grafo aleatorio; imprime o peso e o tempo)
   Prim (1957); Cormen et al. (2009), sec. 23.2; Feofiloff (2020). */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <limits.h>
#include <stdbool.h>
#include <time.h>

typedef struct { int u, v, w; } Aresta;

/* le "n m" e depois m linhas "u v w" (vertices 0..n-1) */
Aresta *le_grafo( int *n, int *m) {
   if (scanf( "%d %d", n, m) != 2) return NULL;
   Aresta *e = malloc( *m * sizeof (Aresta));
   for (int i = 0; i < *m; ++i)
      if (scanf( "%d %d %d", &e[i].u, &e[i].v, &e[i].w) != 3) return NULL;
   return e;
}

/* gerador pseudoaleatorio xorshift (o mesmo em todos os programas do tutorial) */
unsigned long long semente = 88172645463325252ULL;
unsigned aleat( void) {
   semente ^= semente << 13;  semente ^= semente >> 7;  semente ^= semente << 17;
   return (unsigned) (semente >> 11);
}

/* grafo conexo aleatorio: arvore aleatoria + m-(n-1) arestas quaisquer, pesos em 1..10^6 */
Aresta *grafo_aleatorio( int n, int m) {
   Aresta *e = malloc( m * sizeof (Aresta));
   for (int v = 1; v < n; ++v) {
      e[v-1].u = (int) (aleat() % v);  e[v-1].v = v;  e[v-1].w = 1 + aleat() % 1000000;
   }
   for (int i = n - 1; i < m; ++i) {
      int u = aleat() % n, v = aleat() % n;
      if (u == v) v = (v + 1) % n;
      e[i].u = u;  e[i].v = v;  e[i].w = 1 + aleat() % 1000000;
   }
   return e;
}

/* ordem crescente de peso; empate pelas pontas (reproduz a ordem dos slides) */
int cmp_peso( const void *a, const void *b) {
   const Aresta *x = a, *y = b;
   if (x->w != y->w) return x->w < y->w ? -1 : 1;
   if (x->u != y->u) return x->u - y->u;
   return x->v - y->v;
}

double agora( void) { return (double) clock() / CLOCKS_PER_SEC; }

#define INF INT_MAX

/* W[u*n+v] = peso da aresta u-v ou INF. Preenche pai[] e chave[]; a raiz e o vertice 0. */
long prim( int n, const int *W, int pai[], int chave[], bool mostra) {
   bool *dentro = calloc( n, sizeof (bool));
   for (int v = 0; v < n; ++v) { chave[v] = INF;  pai[v] = -1; }
   chave[0] = 0;
   long total = 0;
   for (int it = 0; it < n; ++it) {
      int u = -1;                                  /* o vertice de fora mais barato */
      for (int v = 0; v < n; ++v)
         if (!dentro[v] && (u < 0 || chave[v] < chave[u])) u = v;
      dentro[u] = true;
      total += chave[u];
      if (mostra) printf( "entra %d %d %d\n", u, chave[u], pai[u]);
      for (int v = 0; v < n; ++v)                  /* so os vizinhos de u mudam */
         if (!dentro[v] && W[u*n + v] < chave[v]) {
            chave[v] = W[u*n + v];
            pai[v] = u;
         }
   }
   free( dentro);
   return total;
}

int main( int argc, char *argv[]) {
   int n, m, tempo = argc == 4 && strcmp( argv[1], "tempo") == 0;
   Aresta *e;
   if (tempo) { n = atoi( argv[2]);  m = atoi( argv[3]);  e = grafo_aleatorio( n, m); }
   else if ((e = le_grafo( &n, &m)) == NULL) return 1;
   double t0 = agora();
   int *W = malloc( (size_t) n * n * sizeof (int));
   for (size_t i = 0; i < (size_t) n * n; ++i) W[i] = INF;
   for (int i = 0; i < m; ++i) {                   /* arestas paralelas: fica a mais leve */
      int u = e[i].u, v = e[i].v;
      if (e[i].w < W[u*n + v]) W[u*n + v] = W[v*n + u] = e[i].w;
   }
   int *pai = malloc( n * sizeof (int)), *chave = malloc( n * sizeof (int));
   long p = prim( n, W, pai, chave, !tempo);
   printf( "peso=%ld", p);
   if (tempo) printf( " t=%.6f", agora() - t0);
   printf( "\n");
   return 0;
}

In [ ]:
# Exemplo: Prim a partir de a no grafo condutor (compare com a tabela dos slides).
saida = rodar('prim_matriz.c', entrada=entrada_grafo(9, ARESTAS), mostrar=False)
arvore = []
for lin in saida.splitlines():
    if lin.startswith('entra'):
        v, c, p = map(int, lin.split()[1:])
        print(f'entra {VERT[v]} com chave {c:2d}' + (f' pela aresta ({VERT[p]},{VERT[v]})' if p >= 0 else ''))
        if p >= 0:
            arvore.append((VERT[p], VERT[v]))
print(saida.splitlines()[-1])
assert campo(saida, 'peso') == 37
assert ('b', 'c') in arvore and ('a', 'h') not in arvore, 'Prim escolhe (b,c), não (a,h)'
desenha(arvore, 'Prim a partir de a: a AGM com (b,c)', lado={'a'})
plt.show()

### Exercício 3: Prim com heap

Para grafos **esparsos**, a varredura $\Theta(n)$ por passo é desperdício. Use um **heap mínimo**
de pares $(\mathit{chave}, v)$ na versão "preguiçosa": sempre que a chave de $v$ baixar, insira
um par novo; ao extrair, ignore os pares de vértices que já estão na árvore. O heap e as listas de
adjacência já estão prontos:

- `heap_insere(c, v)`, `heap_remove_min(&c, &v)` e a variável `tam` (número de pares no heap);
- os vizinhos de `u`: `for (int a = primeiro[u]; a != -1; a = prox[a])`, com a outra ponta
  `dest[a]` e o peso `peso[a]`.

Complete `prim(n)`: preencha `pai[]` (com `pai[0] = -1`) e devolva o peso da AGM. O custo deve ser
$O(m \log n)$. O `main` confere o grafo condutor (inclusive o vetor `pai[]` da tabela dos slides)
e 300 grafos aleatórios contra Kruskal.

In [ ]:
%%writefile src/prim_heap.c
/* Exercicio 3: algoritmo de PRIM com heap binario ("preguicoso"): O(m log n).
   Uso: ./prim_heap               (verificacao)
        ./prim_heap tempo n m     (grafo aleatorio; imprime o peso e o tempo)
   TODO: complete prim(). */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <limits.h>
#include <stdbool.h>
#include <time.h>

typedef struct { int u, v, w; } Aresta;

/* le "n m" e depois m linhas "u v w" (vertices 0..n-1) */
Aresta *le_grafo( int *n, int *m) {
   if (scanf( "%d %d", n, m) != 2) return NULL;
   Aresta *e = malloc( *m * sizeof (Aresta));
   for (int i = 0; i < *m; ++i)
      if (scanf( "%d %d %d", &e[i].u, &e[i].v, &e[i].w) != 3) return NULL;
   return e;
}

/* gerador pseudoaleatorio xorshift (o mesmo em todos os programas do tutorial) */
unsigned long long semente = 88172645463325252ULL;
unsigned aleat( void) {
   semente ^= semente << 13;  semente ^= semente >> 7;  semente ^= semente << 17;
   return (unsigned) (semente >> 11);
}

/* grafo conexo aleatorio: arvore aleatoria + m-(n-1) arestas quaisquer, pesos em 1..10^6 */
Aresta *grafo_aleatorio( int n, int m) {
   Aresta *e = malloc( m * sizeof (Aresta));
   for (int v = 1; v < n; ++v) {
      e[v-1].u = (int) (aleat() % v);  e[v-1].v = v;  e[v-1].w = 1 + aleat() % 1000000;
   }
   for (int i = n - 1; i < m; ++i) {
      int u = aleat() % n, v = aleat() % n;
      if (u == v) v = (v + 1) % n;
      e[i].u = u;  e[i].v = v;  e[i].w = 1 + aleat() % 1000000;
   }
   return e;
}

/* ordem crescente de peso; empate pelas pontas (reproduz a ordem dos slides) */
int cmp_peso( const void *a, const void *b) {
   const Aresta *x = a, *y = b;
   if (x->w != y->w) return x->w < y->w ? -1 : 1;
   if (x->u != y->u) return x->u - y->u;
   return x->v - y->v;
}

double agora( void) { return (double) clock() / CLOCKS_PER_SEC; }

#define INF INT_MAX

/* listas de adjacencia: cada aresta u-v aparece duas vezes (u->v e v->u) */
int *primeiro, *prox, *dest, *peso;
int *pai, *chave;
bool *dentro;

/* heap minimo de pares (chave, v); empate: menor v primeiro */
typedef struct { int c, v; } Par;
Par *heap;
int tam;

static bool menor( Par x, Par y) { return x.c < y.c || (x.c == y.c && x.v < y.v); }

void heap_insere( int c, int v) {
   Par x = {c, v};
   int i = tam++;
   while (i > 0 && menor( x, heap[(i-1)/2])) { heap[i] = heap[(i-1)/2];  i = (i-1)/2; }
   heap[i] = x;
}

void heap_remove_min( int *c, int *v) {
   *c = heap[0].c;  *v = heap[0].v;
   Par x = heap[--tam];
   int i = 0;
   for (;;) {
      int f = 2*i + 1;
      if (f >= tam) break;
      if (f + 1 < tam && menor( heap[f+1], heap[f])) ++f;
      if (!menor( heap[f], x)) break;
      heap[i] = heap[f];  i = f;
   }
   heap[i] = x;
}

/* raiz 0; preenche pai[] (pai[0] = -1) e devolve o peso da AGM.
   Use chave[] e dentro[] (ja alocados) e o heap (comeca vazio: tam = 0). */
long prim( int n) {
   /* TODO */
   return 0;
}

/* ---------- verificacao (nao altere) ---------- */
static void monta( int n, int m, Aresta e[]) {
   primeiro = malloc( n * sizeof (int));
   prox = malloc( 2 * m * sizeof (int));  dest = malloc( 2 * m * sizeof (int));
   peso = malloc( 2 * m * sizeof (int));
   pai = malloc( n * sizeof (int));  chave = malloc( n * sizeof (int));
   dentro = malloc( n * sizeof (bool));
   heap = malloc( (2 * m + n + 1) * sizeof (Par));
   for (int v = 0; v < n; ++v) primeiro[v] = -1;
   for (int i = m - 1; i >= 0; --i)             /* de tras para frente: listas na ordem da entrada */
      for (int lado = 0; lado < 2; ++lado) {
         int u = lado ? e[i].v : e[i].u, a = 2*i + lado;
         dest[a] = lado ? e[i].u : e[i].v;  peso[a] = e[i].w;
         prox[a] = primeiro[u];  primeiro[u] = a;
      }
   tam = 0;
}

static void desmonta( void) {
   free( primeiro);  free( prox);  free( dest);  free( peso);
   free( pai);  free( chave);  free( dentro);  free( heap);
}

/* referencia: Kruskal com rotulos */
static long kruskal_ref( int n, int m, Aresta e0[]) {
   Aresta *e = malloc( m * sizeof (Aresta));
   memcpy( e, e0, m * sizeof (Aresta));
   qsort( e, m, sizeof (Aresta), cmp_peso);
   int *comp = malloc( n * sizeof (int));
   for (int v = 0; v < n; ++v) comp[v] = v;
   long p = 0;
   for (int i = 0; i < m; ++i) {
      int a = comp[e[i].u], b = comp[e[i].v];
      if (a == b) continue;
      p += e[i].w;
      for (int x = 0; x < n; ++x) if (comp[x] == b) comp[x] = a;
   }
   free( e);  free( comp);
   return p;
}

/* pai[] precisa descrever uma arvore geradora cujo peso e o devolvido */
static bool arvore_ok( int n, int m, Aresta e[], long p) {
   if (pai[0] != -1) return false;
   long soma = 0;
   for (int v = 1; v < n; ++v) {
      int u = pai[v], melhor = INF;
      if (u < 0 || u >= n) return false;
      for (int i = 0; i < m; ++i)
         if (((e[i].u == u && e[i].v == v) || (e[i].u == v && e[i].v == u)) && e[i].w < melhor)
            melhor = e[i].w;
      if (melhor == INF) return false;
      soma += melhor;
      int x = v, passos = 0;                     /* subir por pai[] precisa chegar a 0 */
      while (x != 0 && passos++ <= n) x = pai[x];
      if (x != 0) return false;
   }
   return soma == p;
}

static int erro( const char *msg) { printf( "ERRO: %s\nok=0\n", msg);  return 0; }

int main( int argc, char *argv[]) {
   if (argc == 4 && strcmp( argv[1], "tempo") == 0) {
      int n = atoi( argv[2]), m = atoi( argv[3]);
      Aresta *e = grafo_aleatorio( n, m);
      double t0 = agora();
      monta( n, m, e);
      long p = prim( n);
      printf( "peso=%ld t=%.6f\n", p, agora() - t0);
      return 0;
   }
   int ex[14][3] = {{0,1,4},{0,7,8},{1,2,8},{1,7,11},{2,3,7},{2,5,4},{2,8,2},
                    {3,4,9},{3,5,14},{4,5,10},{5,6,2},{6,7,1},{6,8,6},{7,8,7}};
   int pai_esperado[9] = {-1, 0, 1, 2, 3, 2, 5, 6, 2};    /* b<-a, c<-b, d<-c, ... (slides) */
   Aresta e[14];
   for (int i = 0; i < 14; ++i) { e[i].u = ex[i][0];  e[i].v = ex[i][1];  e[i].w = ex[i][2]; }
   monta( 9, 14, e);
   long p = prim( 9);
   printf( "peso=%ld pai:", p);
   for (int v = 0; v < 9; ++v) printf( " %d", pai[v]);
   printf( "\n");
   if (p != 37) return erro( "o grafo condutor tem AGM de peso 37");
   for (int v = 0; v < 9; ++v)
      if (pai[v] != pai_esperado[v]) return erro( "pai[] diferente da tabela dos slides");
   desmonta();
   for (int t = 0; t < 300; ++t) {
      int n = 2 + aleat() % 150, m = n - 1 + aleat() % (3 * n);
      Aresta *a = grafo_aleatorio( n, m);
      monta( n, m, a);
      long pp = prim( n);
      if (pp != kruskal_ref( n, m, a)) return erro( "peso diferente de Kruskal num grafo aleatorio");
      if (!arvore_ok( n, m, a, pp)) return erro( "pai[] nao descreve uma arvore geradora com esse peso");
      desmonta();  free( a);
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 3
saida = rodar('prim_heap.c')
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 3 ok!')

In [ ]:
# Visualização: Prim com matriz contra Prim com heap, em grafos esparsos e densos.
ns = [256, 512, 1024, 2048, 4096]
casos = {'esparso (m = 4n)': lambda n: 4 * n, 'denso (m = n²/4)': lambda n: n * n // 4}
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (nome, m_de) in zip(axs, casos.items()):
    for fonte, rot in (('prim_matriz.c', 'matriz: Θ(n²)'), ('prim_heap.c', 'heap: O(m log n)')):
        ts = [campo(rodar(fonte, args=('tempo', n, m_de(n)), mostrar=False), 't') for n in ns]
        ax.plot(ns, ts, 'o-', label=rot)
    ax.set_xscale('log', base=2); ax.set_yscale('log')
    ax.set_xlabel('n'); ax.set_ylabel('tempo (s)'); ax.set_title(nome); ax.legend()
plt.tight_layout(); plt.show()

## 4. Agrupamento com Kruskal

Dado um conjunto de pontos e um número $k$, queremos dividi-los em $k$ grupos de modo que o
**espaçamento** — a menor distância entre dois pontos de grupos diferentes — seja o **maior**
possível. Kleinberg e Tardos (§4.7) mostram que basta rodar Kruskal no grafo completo (peso =
distância) e **parar quando restarem $k$ componentes**; é o mesmo que tirar as $k - 1$ arestas
mais pesadas da AGM. O espaçamento obtido é o peso da próxima aresta que Kruskal aceitaria.

In [ ]:
# Exemplo: três nuvens de pontos no plano.
def nuvens(rng, centros, por_nuvem, espalhamento):
    return [(cx + rng.gauss(0, espalhamento), cy + rng.gauss(0, espalhamento))
            for cx, cy in centros for _ in range(por_nuvem)]

PONTOS = nuvens(random.Random(5), [(0, 0), (8, 1), (3, 8)], 25, 0.8)
xs, ys = zip(*PONTOS)
plt.figure(figsize=(4.5, 4)); plt.scatter(xs, ys, s=15)
plt.title(f'{len(PONTOS)} pontos'); plt.axis('equal'); plt.show()

### Exercício 4: $k$ grupos de espaçamento máximo

Complete `agrupa(pontos, k)`, que devolve `(grupos, espacamento)`: `grupos` é uma lista com o
número do grupo de cada ponto (inteiros de `0` a `k-1`, em qualquer ordem) e `espacamento` é a
menor distância entre pontos de grupos diferentes. Monte a lista de todos os pares
$(\mathit{dist}, i, j)$, ordene e rode Kruskal com um union-find simples em Python até restarem
`k` componentes. Use `math.dist` para a distância.

O teste compara o espaçamento com uma força bruta que examina **todas** as divisões de até 8
pontos em $k$ grupos.

In [ ]:
# Exercício 4: agrupamento de espaçamento máximo
# TODO: implemente agrupa(pontos, k).

def agrupa(pontos, k):
    # TODO: implemente aqui
    pass


# Teste automático
def espacamento(pontos, grupos):
    return min(math.dist(pontos[i], pontos[j])
               for i, j in itertools.combinations(range(len(pontos)), 2) if grupos[i] != grupos[j])

def particoes(n, k):
    """Todas as divisões de range(n) em exatamente k grupos não vazios (rótulos canônicos)."""
    def gera(i, rot, usados):
        if i == n:
            if usados == k:
                yield list(rot)
            return
        for g in range(min(usados + 1, k)):
            rot.append(g)
            yield from gera(i + 1, rot, max(usados, g + 1))
            rot.pop()
    yield from gera(0, [], 0)

rng = random.Random(11)
for _ in range(60):
    n = rng.randint(3, 8)
    k = rng.randint(2, n - 1)
    pts = [(rng.random(), rng.random()) for _ in range(n)]
    grupos, esp = agrupa(pts, k)
    assert len(set(grupos)) == k, f'Devem sair exatamente {k} grupos'
    assert abs(esp - espacamento(pts, grupos)) < 1e-12, 'espacamento não confere com os grupos'
    melhor = max(espacamento(pts, g) for g in particoes(n, k))
    assert abs(esp - melhor) < 1e-12, f'Espaçamento {esp:.4f}, mas o máximo é {melhor:.4f}'
print('Exercício 4 ok!')

In [ ]:
# Visualização: os três grupos das nuvens de pontos (depois de resolver o Exercício 4).
grupos, esp = agrupa(PONTOS, 3)
xs, ys = zip(*PONTOS)
plt.figure(figsize=(4.5, 4)); plt.scatter(xs, ys, c=grupos, cmap='tab10', s=18)
plt.title(f'k = 3, espaçamento = {esp:.2f}'); plt.axis('equal'); plt.show()

## Desafio Final: a AGM euclidiana e o caminho de menor gargalo

Sorteie $n = 300$ pontos no quadrado $[0, 1]^2$ e considere o grafo **completo** em que o peso de
cada aresta é a distância euclidiana.

1. Calcule a AGM com o **Prim de matriz** (é o caso denso: $m = n(n-1)/2$). Você pode gerar a
   entrada para `src/prim_matriz.c` (pesos inteiros: multiplique as distâncias por $10^6$ e
   arredonde) ou reescrever o Prim em Python. Desenhe a árvore sobre os pontos.
2. **Propriedade minimax.** Para um par de vértices $u, v$, o *gargalo* de um caminho é o peso da
   sua aresta mais pesada. Mostre experimentalmente que, para **todo** par $u, v$, o caminho de
   $u$ a $v$ **dentro da AGM** tem o **menor gargalo** entre todos os caminhos de $u$ a $v$ no
   grafo. Para o lado direito, adapte o desafio da aula de Warshall e Floyd (caminho de maior
   gargalo com max/min) para *menor* gargalo com min/max; para o lado esquerdo, faça uma busca
   na árvore a partir de cada vértice guardando a maior aresta do caminho.
3. Explique, com a propriedade do ciclo, **por que** isso tem de valer.

*Para ir além:* implemente o algoritmo de **Borůvka** (1926): enquanto houver mais de um
componente, cada componente escolhe a aresta mais leve que sai dele, e todas entram de uma vez.
Por que ele precisa de no máximo $\lceil \log_2 n \rceil$ rodadas?

In [ ]:
# Desafio: AGM euclidiana e caminho de menor gargalo
# TODO: resolva o desafio abaixo

rng = random.Random(2026)
PTS = [(rng.random(), rng.random()) for _ in range(300)]

# Sua solução aqui

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. Cap. 21 (conjuntos disjuntos) e cap. 23 (§23.1 teorema do corte; §23.2 Kruskal e Prim, Figuras 23.4 e 23.5). (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- KLEINBERG, J.; TARDOS, É. *Algorithm Design*. Addison-Wesley, 2006. §4.5 (AGM, propriedades do corte e do ciclo), §4.6 (union-find), §4.7 (agrupamento).
- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. §9.1 (Prim) e §9.2 (Kruskal).
- DASGUPTA, S.; PAPADIMITRIOU, C.; VAZIRANI, U. *Algorithms*. McGraw-Hill, 2008. §5.1.
- FEOFILOFF, P. *Algoritmos para Grafos (em linguagem C)*. IME-USP. https://www.ime.usp.br/~pf/algoritmos_para_grafos/
- KRUSKAL, J. B. On the Shortest Spanning Subtree of a Graph and the Traveling Salesman Problem. *Proc. AMS*, v. 7, n. 1, p. 48–50, 1956.
- PRIM, R. C. Shortest Connection Networks and Some Generalizations. *Bell System Technical Journal*, v. 36, n. 6, p. 1389–1401, 1957.